# Parameter Sensitivity and Emulator Analysis

In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd

import fates_calibration_library.utils as utils
import fates_calibration_library.clm_functions as clm
import fates_calibration_library.emulator_functions as em
import fates_calibration_library.surface_data_functions as surface
from fates_calibration_library.TFClass import TFEmulator

import tensorflow as tf
from esem.utils import get_random_params

import matplotlib.pyplot as plt
import importlib

2025-07-18 09:57:34.893519: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-18 09:57:34.895248: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-18 09:57:34.921120: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-18 09:57:34.921934: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-18 09:57:37.385940: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT

## Set Up
Load files, set up ensemble information

In [2]:
# dataset with land area 
land_frac_ds_file = os.path.join("/glade/derecho/scratch/afoster/archive",
                            "ctsm60SP_bigleaf_fullgrid/lnd/hist",
                            "ctsm60SP_bigleaf_fullgrid.clm2.h0.0001-02-01-00000.nc")

obs_config_file = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/ilamb_conversion.yaml'
obs_config = utils.get_config_file(obs_config_file)

pft_id_config = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/fates_pft_ids.yaml'
pft_ids = utils.get_config_file(pft_id_config)

# directories
mesh_dir = '/glade/work/afoster/FATES_calibration/mesh_files'
hist_dir = '/glade/work/afoster/FATES_calibration/history_files/compiled_files'
emulator_dir = '/glade/work/afoster/FATES_calibration/emulators'
fig_dir = '/glade/work/afoster/FATES_calibration/figures'
param_dir = '/glade/work/afoster/FATES_calibration/parameter_files'

# grab pft names
default_param = xr.open_dataset(os.path.join(param_dir,
                                             'fates_params_default_sci.1.85.1_api.40.0.0_crops.nc'))
all_pfts = [str(pft).replace("b'", "").replace("'", "").strip() for pft in default_param.fates_pftname.values]
default_norm = pd.read_csv(os.path.join(param_dir, 'normalized_parameters.csv'), index_col=[0])

# variables to emulate
calibration_vars = ['GPP', 'EFLX_LH_TOT', 'FSH', 'EF']

# number of samples to choose
n_sample = 10000

In [3]:
# information about each ensemble
ens_dict = {'dompft':
            {'mesh_file': os.path.join(mesh_dir, 'dominant_grid_mesh.nc'),
             'land_mask_file': os.path.join(mesh_dir, 'dominant_grid.nc'),
             'ensemble_file': os.path.join(hist_dir, 'fates_dompft_annual_means.nc'),
             'lhc_key_file': os.path.join(param_dir, 'fates_lh', 'fates_lh_key.csv'),
             'pfts': [1, 2, 3, 12, 13, 14],
             'obs_df': os.path.join(mesh_dir, 'dominant_grid.csv'),
            }
           }

In [4]:
# choose ensemble
ensemble = 'dompft'

### Load Latin Hypercube Key

In [5]:
lhc_key = pd.read_csv(ens_dict[ensemble]['lhc_key_file'], index_col=[0])
lhc_key = lhc_key.drop(columns=['ensemble'])
param_names = lhc_key.columns
num_params = len(param_names)

### Load Observations

In [6]:
obs = pd.read_csv(ens_dict[ensemble]['obs_df'], index_col=[0])

In [9]:
[all_pfts[12-1]]

['arctic_c3_grass']

## Evaluate Emulators
Loop through each pft and variable to investigate ensemble and emulator spread

In [ ]:
sens_dfs = []
sample_dfs = []
for pft in ens_dict[ensemble]['pfts']:
    
    pft_name = all_pfts[pft-1]
    pft_id = pft_ids[pft_name]
    
    # get observations for this pft
    default_pft = default_norm[default_norm.pft == pft]
    default_pft = default_pft.drop(columns = ['pft'])

    # get observations for this pft
    obs_pft = obs[obs.pft == pft_name]
    obs_pft = obs_pft[obs_pft.land_frac > 0.99]
    obs_pft = obs_pft[obs_pft.pct_lake < 30]

    for variable in calibration_vars:
        
        # get weighted mean and sd for the observations
        obs_mean, obs_sd = em.get_obs_mean_and_sd(obs_pft, obs_config[variable]['var'])

        # load emulator
        model = TFEmulator(emulator_dir, pft=pft_id, variable=variable)

        # generate a random sample and emulate on it
        sample = get_random_params(len(param_names), n_sample)
        pred_sampled, pred_sampled_var = model(sample)

        sample_df = pd.DataFrame(sample)
        sample_df.columns = param_names
        sample_df['pred'] = pred_sampled.numpy().flatten()
        sample_df['pred_var'] = pred_sampled_var.numpy().flatten()
        sample_df['variable'] = variable
        sample_df['pft'] = pft_id
        sample_dfs.append(sample_df)

        # plot histogram of emulated values
        em.plot_emulated_sample(pred_sampled.numpy().flatten(), obs_mean, obs_sd,
                                pft_id, variable, obs_config[variable]['global_units'])
        plt.savefig(os.path.join(fig_dir, 'emulator_figs', f"emulated_sample_{variable}_{pft_id}.png"))

        # sensitivity analysis
        dat = em.plot_oaat_sens(param_names, model, default_pft)
        plt.savefig(os.path.join(fig_dir, 'emulator_figs', f"oaat_sens_{variable}_{pft_id}.png"))

        sens_df = em.sensitivity_analysis(model, param_names)
        sens_df['pft'] = pft_id
        sens_df['variable'] = variable
        sens_dfs.append(sens_df)
        
        em.plot_global_sensitivity(sens_df, variable)
        plt.savefig(os.path.join(fig_dir, 'emulator_figs', f"sobol_sens_{variable}_{pft_id}.png"))
sample_df = pd.concat(sample_dfs)
sens_df = pd.concat(sens_dfs)

In [ ]:
props = {}
sensitive_params = {}
for pft in ens_dict[ensemble]['pfts']:
    
    pft_name = all_pfts[pft-1]
    pft_id = pft_ids[pft_name]

    
    props[pft_id] = {}
    sensitive_params[pft_id] = {}
    
    obs_pft = obs[obs.pft == pft_name]
    obs_pft = obs_pft[obs_pft.land_frac > 0.99]
    obs_pft = obs_pft[obs_pft.pct_lake < 30]
    df = sample_df[sample_df.pft == pft_id]
    pft_sens = sens_df[sens_df.pft == pft_id]
    
    for variable in calibration_vars:
        
        # get weighted mean and sd for the observations
        obs_mean, obs_sd = em.get_obs_mean_and_sd(obs_pft, obs_config[variable]['var'])
        df_var = df[df.variable == variable]
        pft_sens_var = pft_sens[pft_sens.variable == variable]

        # implausibility score
        implaus = em.implausibility_metric(df_var.pred, obs_mean, df_var.pred_var, obs_sd**2)

        em.plot_implausibility_histogram(implaus)

        props[pft_id][variable] = em.get_proportion_implausible(implaus, 1)
        sensitive_params[pft_id][variable] = np.unique(pft_sens_var[pft_sens_var.ST >= 0.01]['parameter'])
sample_props = pd.DataFrame.from_dict(props, orient='index')

In [ ]:
sens_df.to_csv(os.path.join(emulator_dir, f'sensitivity_df_{ensemble}.csv'))
sample_props.to_csv(os.path.join(emulator_dir, f'sample_props_{ensemble}.csv'))

In [ ]:
sensitive_params['BETT']